# 📗 Module 04 · Notebook 00 — Transformer from Scratch

Sebelum memakai LLM siap pakai, kita **bangun mesinnya dari nol**: tokenizer, embedding,
**self-attention**, transformer block, lalu rakit **GPT mungil** dan **latih** sampai ia
"belajar" pola bahasa Indonesia. Semua pakai **PyTorch murni** (tanpa `transformers`).

## Apa yang akan kita pelajari?
1. Tokenization & embedding dari nol
2. **Self-attention** (scaled dot-product) — inti Transformer
3. Causal mask + multi-head attention
4. Transformer block (LayerNorm, GELU, feed-forward, residual)
5. Rakit GPT mungil & latih (next-token prediction)
6. Generate teks & lihat ia belajar

> Ini "GPT" yang sama secara prinsip dengan raksasa modern — bedanya skala & data.

In [ ]:
# PyTorch murni — tidak perlu transformers di notebook ini
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Hyperparameter tiny GPT (cukup kecil untuk dilatih dalam hitungan menit di T4)
batch_size = 32      # banyak sequence paralel per langkah
block_size = 64      # panjang konteks maksimum (token)
n_embd = 128         # dimensi embedding
n_head = 4           # jumlah attention head
n_layer = 4          # jumlah transformer block
dropout = 0.1
learning_rate = 3e-4
max_iters = 3000
eval_interval = 300
eval_iters = 100

## 1. Korpus & Tokenization (char-level)

Model tidak membaca huruf — ia membaca **angka**. Tokenizer mengubah teks ↔ ID.
Di sini kita pakai **char-level** (tiap karakter = 1 token) agar sederhana; LLM nyata
memakai **subword (BPE)**. Kita latih di korpus kecil berbahasa Indonesia.

In [ ]:
text = """Kecerdasan buatan mengubah cara manusia bekerja dan belajar.
Model bahasa besar dilatih dari teks dalam jumlah masif di internet.
Sebuah model belajar memprediksi kata berikutnya dari konteks sebelumnya.
Semakin banyak data dan parameter, semakin kaya pola yang bisa ditangkap.
Transformer memproses kata secara paralel lewat mekanisme attention.
Attention membuat model menimbang kata mana yang paling relevan.
Dengan begitu model memahami konteks kalimat yang panjang sekalipun.
Pembelajaran mesin adalah fondasi dari kecerdasan buatan modern.
Jaringan saraf tiruan terinspirasi dari cara kerja otak manusia.
Data yang berkualitas sama pentingnya dengan ukuran model itu sendiri.
"""
# Gandakan & sambung agar korpus cukup panjang untuk melatih pola char-level.
text = (text + "\n") * 60

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("panjang korpus (char):", len(text))
print("ukuran vocab:", vocab_size)
print("contoh encode 'data':", encode("data"))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## 2. Embedding & Self-Attention

Tiap token ID → **vektor** (token embedding). Kita tambah **positional embedding** agar model
tahu urutan. Lalu inti Transformer: **self-attention** — tiap posisi menghitung Query, Key,
Value; skor `Q·Kᵀ/√d` → softmax → rata-rata berbobot atas Value.

In [ ]:
# Intuisi attention pada satu contoh kecil (B=1, T=4, C=2)
torch.manual_seed(0)
B, T, C = 1, 4, 2
x = torch.randn(B, T, C)
q = k = v = x                      # sederhana: Q=K=V=x untuk ilustrasi
wei = q @ k.transpose(-2, -1) * C ** -0.5   # (B,T,T) skor kemiripan
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))   # causal: tak boleh lihat masa depan
wei = F.softmax(wei, dim=-1)
print("bobot attention (tiap baris jumlahnya 1, segitiga bawah):")
print(wei[0])
out = wei @ v
print("output shape:", out.shape)

In [ ]:
class Head(nn.Module):
    """Satu attention head (causal)."""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v                                         # (B,T,head_size)

## 3. Multi-Head Attention & Transformer Block

Beberapa head berjalan paralel (tiap head "memperhatikan" hal berbeda), lalu disatukan.
Satu **transformer block** = multi-head attention + feed-forward, dibungkus **LayerNorm**
(pre-norm) dan **koneksi residual** agar gradien mengalir mulus.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: pre-norm + residual."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 4. Rakit GPT Mungil

Tumpuk `n_layer` block + LayerNorm akhir + `lm_head` (proyeksi ke vocab). Tambah method
`generate` untuk sampling autoregresif.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)                       # (B,T,C)
        pos = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                    # (B,T,vocab)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]                         # potong ke konteks
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]                              # token terakhir
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

model = TinyGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"jumlah parameter: {n_params:,} (~{n_params / 1e6:.2f} jt)")

## 5. Sebelum Dilatih → Latih → Sesudah

Model yang baru diinisialisasi menghasilkan **acak**. Kita latih (next-token prediction),
lalu generate lagi — perhatikan teks berubah dari acak menjadi mirip pola bahasa Indonesia.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("=== SEBELUM training (acak) ===")
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for it in range(max_iters + 1):
    if it % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {it:4d} | train loss {losses['train']:.3f} | val loss {losses['val']:.3f}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
print("=== SESUDAH training ===")
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))

## Ringkasan & Jembatan

Kita membangun Transformer dari nol: tokenization → embedding → **self-attention** →
multi-head → block → GPT, lalu **melatihnya** hingga loss turun dan output makin mirip bahasa.

| Komponen | Fungsi |
|----------|--------|
| Token + positional embedding | teks → vektor + info urutan |
| Self-attention | timbang token relevan (Q·Kᵀ/√d → softmax → V) |
| Causal mask | cegah melihat token masa depan |
| Multi-head | banyak "sudut perhatian" paralel |
| Transformer block | attention + FFN + LayerNorm + residual |

GPT raksasa (GPT-4, Qwen, Llama) = **prinsip yang sama**, tapi miliaran parameter + data
internet + ribuan GPU. **Notebook 01**: kita pakai salah satu yang sudah dilatih.